## Data Read

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

### Data Load

In [0]:
# Bronze layer path
bronze_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/"

old_battery_data = f"{bronze_path}old_battery_data"
realtime_ingestion = f"{bronze_path}realtime_ingestion"
bms_logs = f"{bronze_path}bms_logs"
realtime_bms_logs = f"{bronze_path}realtime_bms_logs"

### Spark Dataframe Create

In [0]:
df_old_battery = spark.read.format('parquet')\
               .option('inferScehma', True)\
               .load(old_battery_data)

df_realtime_battery = spark.read.format('parquet')\
               .option('inferScehma', True)\
               .load(realtime_ingestion)

df_old_bms = spark.read.format('text')\
               .option('inferScehma', True)\
               .load(bms_logs)

df_realtime_bms = spark.read.format('text')\
               .option('inferScehma', True)\
               .load(realtime_bms_logs)

### Schema Enforcement: Historical battery data

In [0]:
df_old_battery.describe()

df_old_battery.toPandas().info()

Type Casting

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType, DoubleType, IntegerType
# Step 1: Schema Enforcing

df_silver_battery_olddata = df_old_battery.select(
            col("timestamp").cast(TimestampType()).alias("timestamp"),
            col("voltage").cast(DoubleType()).alias("voltage"),
            col("temperature").cast(DoubleType()).alias("temperature"),
            col("cycle_count").cast(DoubleType()).cast(IntegerType()).alias("cycle_count"),
            col("battery_id")
)

df_silver_battery_olddata.toPandas().info()

In [0]:
df_silver_battery_olddata.display()

### Schema Enforcement: Realtime battery data

In [0]:
df_realtime_battery.display()

In [0]:
df_realtime_battery.printSchema()

In [0]:
df_realtime_body = df_realtime_battery.select(col("body"))
df_realtime_body.display()

Schema Generate

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

battery_data_schema = StructType([
    StructField("battery_id", StringType(), True),
    StructField("voltage", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temp", DoubleType(), True),
    StructField("cycle_count", IntegerType(), True)
])

Data Flattening

In [0]:
# parse json string into json
df_realtime_battery = df_realtime_battery.withColumn("parsed_body", from_json(col("body"), battery_data_schema))


df_silver_realtime_battery = df_realtime_battery.select(col("parsed_body.battery_id").alias("battery_id"),
                                                        col("parsed_body.voltage").alias("voltage"),
                                                        col("parsed_body.timestamp").alias("timestamp"),
                                                        col("parsed_body.temp").alias("temp"),
                                                        col("parsed_body.cycle_count").alias("cycle_count"))
df_silver_realtime_battery.toPandas().info()                                                      

### Schema Enforcement: Old BMS error logs

In [0]:
df_old_bms.display()

df_old_bms.toPandas().info()

### Schema Enforcement: Realtime BMS error logs

In [0]:
df_realtime_bms.display()

df_realtime_bms.toPandas().info()